<a href="https://colab.research.google.com/github/diarce/Auditoria/blob/main/AuditMayorista_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AuditMayorista 6.0.0 — Auditoría digital del desarrollo funcional del comercio electrónico

**Tesis:** *Factores estratégicos asociados al desarrollo del comercio electrónico en tiendas mayoristas de bienes de consumo masivo de Posadas, Misiones*  
**Autor:** Diego Enrique Arce · **Director:** Dr. Carlos Roberto Brys · UNaM, FCE, Maestría en Administración Estratégica de Negocios

Este cuaderno ejecuta el instrumento descripto en el capítulo 7 y los anexos B y C: 32 indicadores en 8 dimensiones (25 automáticos y 7 manuales), escala 0-3 y comparación de perfiles por dimensión, sin índice agregado.

> **Antes de guardar este cuaderno en GitHub:** *Editar → Borrar todas las salidas*. Las salidas pueden contener identificadores internos del estudio.

## ① Configuración

In [ ]:
import os, subprocess, sys

REPO = "https://github.com/diarce/Auditoria.git"
DIR = "/content/Auditoria"
if os.path.isdir(DIR):
    subprocess.run(["git", "-C", DIR, "pull", "--quiet"], check=True)
else:
    subprocess.run(["git", "clone", "--quiet", REPO, DIR], check=True)
os.chdir(DIR)
sys.path.insert(0, DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

import config
commit = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
print(f"AuditMayorista {config.VERSION} · commit {commit}")
print("Contacto informado a los sitios:", config.CONTACTO_INVESTIGADOR)

## ② Universo del estudio (archivo privado)

Cargue `universo.csv` (mismo formato que `data/universo_ejemplo.csv`). El archivo se guarda en `data/privado/`, carpeta que no se publica. Sin ese archivo, el cuaderno muestra solo el universo de ejemplo y no audita.

In [ ]:
from pathlib import Path
from IPython.display import HTML, display
from config import PRIVATE_DIR, UNIVERSO_PRIVADO

try:
    from google.colab import files
    if not UNIVERSO_PRIVADO.exists():
        subido = files.upload()                       # seleccionar universo.csv
        PRIVATE_DIR.mkdir(parents=True, exist_ok=True)
        UNIVERSO_PRIVADO.write_bytes(next(iter(subido.values())))
except ImportError:
    pass                                              # fuera de Colab: copiar el archivo a data/privado/

from modules.universo import cargar_universo, sitios_para_auditar, tabla_universo_html, usa_datos_de_ejemplo
universo = cargar_universo()
display(HTML(tabla_universo_html(universo)))
sitios = sitios_para_auditar(universo)
print(f"Plataformas con canal de venta propio dentro del ámbito: {len(sitios)}")

## ③ Auditoría automatizada (25 indicadores)

Antes de cada solicitud se consulta `robots.txt`. Si no autoriza la página de inicio, el listado o la ficha de producto, la plataforma se **excluye** y la exclusión se informa como resultado (sección 7.4). Las secciones secundarias bloqueadas y las páginas detrás de un inicio de sesión se valoran 0.

In [ ]:
import logging
logging.basicConfig(level=logging.WARNING)
from config import DB_PATH
from modules.auditor import AuditEngine, DESCRIPCION_EXCLUSION
from modules.storage import DatabaseManager

db = DatabaseManager(DB_PATH)
if usa_datos_de_ejemplo():
    print("No hay universo privado: no se audita el universo de ejemplo.")
else:
    resultado = AuditEngine(db).run(sites_override=sitios)
    for sid, r in resultado.items():
        print(sid, "→", f"EXCLUIDA: {DESCRIPCION_EXCLUSION[r.codigo_exclusion]}" if r.excluida
              else f"{len(r.resultados)} indicadores valorados")

## ④ Verificación manual (7 indicadores)

Genere la planilla, complétela con la verificación manual de cada plataforma auditada y vuelva a cargarla. Los valores quedan registrados con método «M».

In [ ]:
from modules.importer import AuditImporter, generar_plantilla_csv
ids = sorted({r["site_id"] for r in db.get_audit_results()})
plantilla = generar_plantilla_csv(ids)
print("Planilla:", plantilla)
# files.download(str(plantilla))          # descargar para completar

In [ ]:
# Cargar la planilla completada
# subido = files.upload()
# ruta = PRIVATE_DIR / next(iter(subido)); ruta.write_bytes(next(iter(subido.values())))
# res = AuditImporter(db).from_csv(ruta)
# print(res["insertadas"], "valores cargados;", res["errores"][:5])

## ⑤ Concordancia entre codificación automática y manual

Se extrae una muestra aleatoria estratificada de valores automáticos (semilla fija). Un codificador la valora **sin conocer** el valor automático, con las reglas de `docs/reglas_de_valoracion.md`. Fije y cite el criterio de interpretación del coeficiente kappa antes de conocer el resultado.

In [ ]:
from modules.concordancia import calcular_concordancia, generar_muestra_control, importar_control
if db.get_audit_results():
    print("Muestra:", generar_muestra_control(db, n=40, semilla=2026))
# Luego de completarla:
# print(importar_control(db, PRIVATE_DIR / "muestra_control_semilla2026.csv", codificador="Codificador 1"))
# print(calcular_concordancia(db))

## ⑥ Resultados (sección 7.4.2)

Distribución de frecuencias, puntaje por dimensión y comparación de perfiles. Las plataformas se identifican con alias.

In [ ]:
from modules.reporter import CSS, DatosInforme, HTMLReporter, svg_distribucion, svg_perfiles
d = DatosInforme(db)
if d.results:
    display(HTML(CSS + '<div class="viz-root">' + svg_distribucion(d.results) + svg_perfiles(d.por_sitio, d.alias) + "</div>"))
print(f"Plataformas auditadas: {len(d.auditados)} · excluidas: {len(d.exclusiones)}")

In [ ]:
informe = HTMLReporter(db).generate()
print("Informe:", informe)
# files.download(str(informe))

## Cita

```bibtex
@software{AuditMayorista2026,
  author  = {Arce, Diego Enrique},
  title   = {{AuditMayorista}: instrumento de auditoría digital automatizada del desarrollo
             funcional de plataformas de comercio electrónico mayorista},
  year    = {2026},
  version = {6.0.0},
  url     = {https://github.com/diarce/Auditoria}
}
```
Cite también el commit que muestra la celda ①.